In [46]:
# ====================== 配置区 ======================
from pathlib import Path
import pandas as pd, numpy as np, json, ast

ROOT = Path("./results")
SETTING = "gcd"   # "gcd" | "openset"

# 只纳入的 dataset（不存在自动跳过）
DATASETS_WHITELIST = [
    "20NG","DBPedia","TREC","Yahoo",
    "banking","clinc","ecdt","ele","hwu","mcid",
    "news","stackoverflow","thucnews",
]

# 指标
METRICS_MAP = {
    "gcd":     ["K-ACC", "N-ACC"],
    "openset": ["K-F1", "N-F1"],
}

# LCR 固定值（按 setting）
LCR_KEEP = {"gcd": 0.1, "openset": 1.0}

# 固定 KCR
KCR_FILTER = 0.25

# 方法排序
# METHOD_ORDER_MAP = {
#     "gcd": ["DeepAligned", "GeoID", "SDC", "DPN", "TAN", "LOOP", "ALUP"],
#     "openset": [
#         "DOC", "DeepUNK", "ADB", "SCL", "AB", "KNNCon", "DyEn",
#         "MaxSoftmax", "MaxLogit", "EnergyBased", "Entropy",
#         "LogitNorm", "TemperatureScaling", "OpenMax", "Mahalanobis", "KLMatching",
#     ],
# }
METHOD_ORDER_MAP = {
    "gcd": ["DeepAligned", "GeoID", "SDC", "DPN", "TAN", "LOOP", "ALUP"],
    "openset": ["DOC", "DeepUNK", "ADB", "SCL", "AB", "KNNCon", "DyEn", 
                 "OpenMax",  
                "EnergyBased",  "MaxLogit"
            ], 
}

# 名称归一化/重命名（确保展示名与 METHOD_ORDER_MAP 完全一致）
METHOD_RENAME = {
    # gcd
    "deepaligned":"DeepAligned","geoid":"GeoID","sdc":"SDC","dpn":"DPN","tan":"TAN","loop":"LOOP","alup":"ALUP",
    # openset 常规法
    "doc":"DOC","deepunk":"DeepUNK","adb":"ADB","scl":"SCL","ab":"AB",
    "knncon":"KNNCon","knn-con":"KNNCon","knn_con":"KNNCon","dyen":"DyEn",
    "plm_ood":"LLM-OOD"
}

# 采样分布显示顺序（不确定就留空，程序会按出现顺序/字母序）
FOLDTYPE_ORDER = []      # 例如：["fold","iid","niid","random"]
FOLD_TYPES_KEEP = None   # 例如只看 fold: ["fold"]

ROUND_DECIMALS = 2
HIGHLIGHT_TOP2 = True
OUTPUT_TEX = f"{SETTING}_avg_by_foldtype_KCR{KCR_FILTER:.2f}.tex"
# ==================== 配置区结束 ====================

In [47]:
# ------------------ 工具函数 ------------------
def _normalize_key(s: str) -> str:
    return str(s).strip().lower().replace("_","").replace("-","")

def rename_method(raw_name: str, fallback_task: str) -> str:
    key = _normalize_key(raw_name if pd.notnull(raw_name) else fallback_task)
    return METHOD_RENAME.get(key, str(raw_name if pd.notnull(raw_name) else fallback_task))

def parse_args_field(s: str) -> dict:
    if pd.isna(s): return {}
    txt = str(s)
    try:
        return json.loads(txt)
    except Exception:
        pass
    try:
        fixed = txt.replace('""','"')
        if len(fixed)>=2 and fixed[0]=='"' and fixed[-1]=='"':
            fixed = fixed[1:-1]
        return json.loads(fixed)
    except Exception:
        pass
    try:
        return ast.literal_eval(txt)
    except Exception:
        return {}

def latex_escape_text(x) -> str:
    s = str(x)
    rep = {"\\": r"\textbackslash{}", "&": r"\&", "%": r"\%", "$": r"\$", "#": r"\#",
           "_": r"\_", "{": r"\{", "}": r"\}", "~": r"\textasciitilde{}", "^": r"\textasciicircum{}"}
    return "".join(rep.get(ch, ch) for ch in s)

In [48]:
# ------------------ 读取与标准化（含 plm_ood→Detector） ------------------
base = ROOT / SETTING
metrics = METRICS_MAP[SETTING]
ALIASES_PLM = {"plmood","llmood"}

task_list = sorted(p.name for p in base.iterdir() if p.is_dir() and (p/"results.csv").exists())
frames, missing = [], []
for task in task_list:
    csv_path = base / task / "results.csv"
    try:
        tdf = pd.read_csv(csv_path)
    except Exception as e:
        print(f"[WARN] 读取失败，已跳过：{csv_path}  |  错误：{type(e).__name__}: {e}")
        missing.append(str(csv_path))
        continue

    # 解析 args 一次，复用 fold_type / Detector
    tdf["__args_obj__"] = tdf["args"].apply(parse_args_field)
    tdf["FoldType"] = tdf["__args_obj__"].apply(lambda d: d.get("fold_type") or d.get("foldType") or d.get("fold-type"))

    # （可选）只保留某些fold_type
    if FOLD_TYPES_KEEP:
        tdf = tdf[tdf["FoldType"].astype(str).str.lower().isin([x.lower() for x in FOLD_TYPES_KEEP])].copy()
        if tdf.empty:
            continue

    # 统一 Method：一般法重命名；plm_ood → Detector/Detecor
    if "method" in tdf.columns:
        def _method_from_row(row):
            raw_m = row.get("method", None)
            renamed = rename_method(raw_m, task)
            if any(_normalize_key(x) in ALIASES_PLM for x in (task, raw_m, renamed)):
                det = row["__args_obj__"].get("Detector") or row["__args_obj__"].get("Detecor")
                return det if det else "LLM-OOD"
            return renamed
        tdf["Method"] = tdf.apply(_method_from_row, axis=1)
    else:
        if _normalize_key(task) in ALIASES_PLM:
            tdf["Method"] = tdf["__args_obj__"].apply(lambda d: d.get("Detector") or d.get("Detecor") or "LLM-OOD")
        else:
            tdf["Method"] = rename_method(task, task)

    # 基本字段
    tdf["dataset"] = tdf["dataset"].astype(str)
    tdf["KCR"] = tdf.get("known_cls_ratio", np.nan)
    tdf["LCR"] = tdf.get("labeled_ratio", np.nan)

    frames.append(tdf)

if not frames:
    raise FileNotFoundError("未找到任何 results.csv；失败文件：\n" + "\n".join(missing))

df = pd.concat(frames, ignore_index=True)

# —— 仅保留白名单方法（未列出的直接不展示）
ALLOWED = [m for m in METHOD_ORDER_MAP.get(SETTING, []) if m]
if ALLOWED:
    df["Method"] = df["Method"].astype(str)
    df = df[df["Method"].isin(set(ALLOWED))].copy()
    if df.empty:
        raise ValueError(
            f"{SETTING}: 方法白名单过滤后无数据。请检查 METHOD_ORDER_MAP['{SETTING}'] "
            f"是否包含实际的 Method 名（openset 下也要包含各 Detector 名）。"
        )

# 白名单数据集
wl = [d.lower() for d in DATASETS_WHITELIST]
df = df[df["dataset"].str.lower().isin(wl)].copy()
if df.empty:
    raise ValueError("白名单筛选后无数据。")

# 过滤 KCR / LCR
df = df[(df["KCR"] == KCR_FILTER) & (df["LCR"] == LCR_KEEP[SETTING])].copy()
if df.empty:
    raise ValueError(f"过滤后无数据：KCR={KCR_FILTER}, LCR={LCR_KEEP[SETTING]}。")

# openset 值<1 视作分数×100
if SETTING == "openset":
    for m in metrics:
        if m in df.columns:
            df[m] = df[m].apply(lambda v: (v*100.0) if (pd.notnull(v) and isinstance(v,(int,float)) and v<1.0) else v)

# ------------------ 聚合：先 (dataset, Method, FoldType) 均值，再跨数据集均值 ------------------
required_cols = ["dataset","Method","FoldType"] + metrics
miss = [c for c in required_cols if c not in df.columns]
if miss:
    raise KeyError(f"缺失列：{miss}")

g1 = (df[required_cols]
      .groupby(["dataset","Method","FoldType"], as_index=False)
      .mean(numeric_only=True))

g2 = (g1.groupby(["FoldType","Method"], as_index=False)
         .mean(numeric_only=True))  # 跨数据集的平均

# ------------------ 排序 ------------------
# FoldType 顺序
present_ft = g2["FoldType"].dropna().astype(str).unique().tolist()
if FOLDTYPE_ORDER:
    ft_order = [x for x in FOLDTYPE_ORDER if x in present_ft] + sorted([x for x in present_ft if x not in FOLDTYPE_ORDER])
else:
    ft_order = sorted(present_ft)  # 没指定就按字母序
g2["FoldType"] = pd.Categorical(g2["FoldType"], categories=ft_order, ordered=True)

# Method 顺序
present_methods = set(g2["Method"].astype(str).unique().tolist())
whitelist = METHOD_ORDER_MAP.get(SETTING, []) or []
method_order = [m for m in whitelist if m in present_methods]  # 只按白名单
g2["Method"] = pd.Categorical(g2["Method"], categories=method_order, ordered=True)
g2 = g2.dropna(subset=["Method"])  # 保险：去掉不在白名单中的行
g2 = g2.sort_values(["FoldType","Method"]).reset_index(drop=True)

# ------------------ 透视（行=Method；列=(FoldType, Metric)） ------------------
metrics_to_use = METRICS_MAP[SETTING]

# 行：Method；列：FoldType×Metric
pivot = g2.pivot_table(index="Method",
                       columns="FoldType",
                       values=metrics_to_use,
                       aggfunc="mean")

# 规范列顺序：按 ft_order 与 metrics_to_use
# pivot.columns 是一个 MultiIndex: (Metric, FoldType)
# 我们要把它重排为 (FoldType, Metric)
pivot = pivot.reorder_levels([1, 0], axis=1)
existing = set(map(tuple, pivot.columns))
ordered_cols = [(ft, m) for ft in ft_order for m in metrics_to_use if (ft, m) in existing]
pivot = pivot.reindex(columns=pd.MultiIndex.from_tuples(ordered_cols))

# 数值格式
pivot = pivot.round(ROUND_DECIMALS)

# ------------------ 高亮：每个 (FoldType, Metric) 列内对 Method 排名 ------------------
def highlight_col(col_ser: pd.Series) -> pd.Series:
    out = col_ser.astype(object)
    ser = pd.to_numeric(col_ser, errors="coerce")
    order = ser.sort_values(ascending=False, kind="mergesort")
    if len(order) >= 1 and pd.notnull(order.iloc[0]):
        out.loc[order.index[0]] = "\\textbf{" + f"{order.iloc[0]:.{ROUND_DECIMALS}f}" + "}"
    if len(order) >= 2 and pd.notnull(order.iloc[1]):
        out.loc[order.index[1]] = "\\underline{" + f"{order.iloc[1]:.{ROUND_DECIMALS}f}" + "}"
    for idxi, v in ser.items():
        if pd.isnull(v):
            out.loc[idxi] = ""
        elif out.loc[idxi] == col_ser.loc[idxi]:
            out.loc[idxi] = f"{v:.{ROUND_DECIMALS}f}"
    return out

formatted = pd.DataFrame(index=pivot.index)
for col in pivot.columns:
    formatted[col] = highlight_col(pivot[col])

# ------------------ LaTeX（三层表头：FoldType → Metric；行：Method） ------------------
# 列规格：左侧 Method 一列；每个 FoldType 组内有 len(metrics_to_use) 列，并以竖线分隔组
group_specs = []
for ft in ft_order:
    n = sum(1 for (f, m) in ordered_cols if f == ft)
    if n > 0:
        group_specs.append("r" * n)
colspec = "l|" + "|".join(group_specs) if group_specs else "l"

# FoldType 显示名（转义下划线等）
def disp_ft(s: str) -> str:
    return latex_escape_text(str(s))

# 表头第1行：FoldType 组
head_line1 = " & ".join(
    ["Method"] +
    [f"\\multicolumn{{{sum(1 for (f,m) in ordered_cols if f==ft)}}}{{c{'|' if i < len(group_specs)-1 else ''}}}{{{disp_ft(ft)}}}"
     for i, ft in enumerate([ft for ft in ft_order if any(f==ft for (f,_) in ordered_cols)])]
) + " \\\\"

# 表头第2行：Metric
head_line2 = " & ".join(
    [""] + [m for (ft, m) in ordered_cols]
) + " \\\\ \\midrule"

def row_to_latex(method, row):
    method_str = latex_escape_text(method)
    cells = [method_str]
    for ft, m in ordered_cols:
        v = row.get((ft, m), "")
        cells.append(v if isinstance(v, str) else ("" if pd.isnull(v) else f"{v:.{ROUND_DECIMALS}f}"))
    return " & ".join(cells) + " \\\\"

body_lines = [row_to_latex(idx, formatted.loc[idx]) for idx in formatted.index]

# 安全版 resizebox 括号
env = "table*"; size_cmd = "\\small"; tabcol = 4
latex_table = "\n".join([
    f"\\begin{{{env}}}[t]",
    "\\centering",
    size_cmd,
    f"\\setlength{{\\tabcolsep}}{{{tabcol}pt}}",
    "\\resizebox{\\textwidth}{!}{%",
    f"\\begin{{tabular}}{{{colspec}}}",
    "\\toprule",
    head_line1,
    head_line2,
    "\n".join(body_lines),
    "\\bottomrule",
    "\\end{tabular}%",
    "}",
    f"\\end{{{env}}}"
])

Path(OUTPUT_TEX).write_text(latex_table, encoding="utf-8")
print(f"[OK] 采样分布平均表生成：{Path(OUTPUT_TEX).resolve()}")
print("FoldType 顺序：", ft_order)

[OK] 采样分布平均表生成：/ssd/code/bolt/gcd_avg_by_foldtype_KCR0.25.tex
FoldType 顺序： ['fold', 'imbalance_fold']


/tmp/ipykernel_4118433/2386170147.py:117: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot = g2.pivot_table(index="Method",
